In [1]:
from langgraph.graph import StateGraph, START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Literal
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [3]:
class SentimentSchema(BaseModel):
    sentiments : Literal["positive","negative"] = Field(description="Sentiment of review")

In [4]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [5]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)


In [6]:
result = structured_model.invoke(
    "This product is absolutely amazing!"
)

print(result)
#print(result.sentiment)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


sentiments='positive'


In [38]:
class ReviewState(TypedDict):
    review : str
    sentiments: Literal["positive","negative"]
    diagnosis : dict
    response : str

In [ ]:
def find_sentiment(state : ReviewState):
    prompt = f'For the following review find the sentiments /n {state["review"]}'

    sentiment = structured_model.invoke(prompt).sentiments
    return{
        "sentiments" : sentiment
    }



def check_sentiments(state : ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiments'] == 'positive':
        return 'positive_response'

    else: 
        return 'run_diagnosis'

In [40]:
def positive_response(state : ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review' ]}\"\n
    Also, kindly ask the user to leave feedback on our website. """

    response = model.invoke(prompt).content

    return {'response' : response} 


In [ ]:
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}
    

In [42]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiments)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [43]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiments': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': "Subject: Good News! Resolution for your Urgent Bug Report - [Your Ticket/Issue ID]\n\nHi [User Name],\n\nI'm truly sorry to hear you've been dealing with such a frustrating bug, and I completely understand why this was marked as high urgency. We know how disruptive these issues can be, and your frustration is absolutely valid.\n\nThank you for bringing this to our attention so quickly. Our team immediately prioritized your report, and I'm pleased to let you know that **we've identified the root cause and successfully deployed a fix for the bug you encountered.**\n\nTo confirm everything is now working as expected for you, could 